In [3]:
!git clone https://github.com/SupercalifragilisticexpialidociousPoppy/OmniEyes.git

Cloning into 'OmniEyes'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 97 (delta 43), reused 83 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 504.45 KiB | 2.39 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [4]:
%cd /content/OmniEyes

/content/OmniEyes


In [5]:
!pip -q install -U transformers accelerate
!pip -q install decord sentence-transformers scipy

In [ ]:
!git clone https://github.com/Lucky-Lance/PhoStream
!hf download lucky-lance/PhoStream phone_class_reencoded_2fps.tar.gz --repo-type dataset --local-dir videos
!cd videos && tar -xf phone_class_reencoded_2fps.tar.gz

Cloning into 'PhoStream'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 16 (delta 0), reused 16 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 845.16 KiB | 13.42 MiB/s, done.
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.

phone_class_reencoded_2fps.tar.gz: downloading bytes:  12% 246M/2.08G [00:02<00:09, 200MB/s, 20.9MB/s  ]
phone_class_reencoded_2fps.tar.gz: reconstructing file:   6% 134M/2.08G [00:03<00:44, 44.1MB/s]
phone_class_reencoded_2fps.tar.gz: downloading bytes:  22% 449M/2.08G [00:04<00:07, 208MB/s, 34.2MB/s  ]
phone_class_reencoded_2fps.tar.gz: downloading bytes:  27% 563M/2.08G [00:04<00:05, 255MB/s, 43.7MB/s  ]
phone_class_reencoded_2fps.tar.gz: downloading bytes:  29% 607M/2.08G [00:04<00:06, 239MB/s, 47.2MB/s  ]
phone_class_reencoded_2fps.tar.gz: downloading bytes:  57% 1.18G/2.08

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/OmniEyes/iteration3

Mounted at /content/drive


In [ ]:
!python src/run.py --ann PhoStream/dataset/phostream_annotations.json --video_root videos \
  --n 3 --taus 0.90 --tasks instant backward \
  --out /content/drive/MyDrive/OmniEyes/iteration3/omni_smoke.jsonl

2770 eligible QAs; running 3
Loading weights: 100% 1442/1442 [00:38<00:00, 37.38it/s] 
[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audi

In [ ]:
import json
from collections import defaultdict
rows = defaultdict(list)
for l in open('/content/drive/MyDrive/OmniEyes/iteration3/omni_smoke.jsonl'):
    r = json.loads(l); rows[r['qid']].append(r)

for qid, rs in rows.items():
    print('=' * 100)
    print(f"[{rs[0]['task']}] Q:   {rs[0]['question']}")
    print(f"        REF: {rs[0]['answer']}")
    for r in rs:
        kept = 'all 33' if r['cond'] == 'full' else r['kept']
        print(f"  {r['cond']:<12} kept={str(kept):<16} {r['gen_tokens']:>3} tok  {r['llm_latency_s']:.2f}s | {r['pred']}")

[instant] Q:   Does the file path update immediately after dropping the icon?
        REF: Yes, the location field in the save dialog instantly changes to "Secret Project," which matches the name of the folder that was dragged in.
  full         kept=all 33            15 tok  3.45s | No, the file path does not update immediately after dropping the icon.
  gate@0.9     kept=[0]               15 tok  0.96s | No, the file path does not update immediately after dropping the icon.
  uniform@0.9  kept=[0]               15 tok  0.95s | No, the file path does not update immediately after dropping the icon.
  pixel@0.9    kept=[0]               15 tok  0.99s | No, the file path does not update immediately after dropping the icon.
[instant] Q:   What exercise is the person doing at the gym?
        REF: After her cardio warm-up, the video shows the person at the gym using a leg press machine for strength training.
  full         kept=all 33            15 tok  3.20s | The person is not doing an e

In [ ]:
!python src/run.py --ann PhoStream/dataset/phostream_annotations.json --video_root videos \
  --n 3 --taus 0.90 --tasks instant backward --keep_current --min_keep 4 \
  --out /content/drive/MyDrive/OmniEyes/iteration3/omni_smoke_fixed.jsonl

2770 eligible QAs; running 3
Loading weights: 100% 1442/1442 [00:38<00:00, 37.37it/s]
[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio

In [ ]:
import json
from collections import defaultdict
rows = defaultdict(list)
for l in open('/content/drive/MyDrive/OmniEyes/iteration3/omni_smoke_fixed.jsonl'):
    r = json.loads(l); rows[r['qid']].append(r)

for qid, rs in rows.items():
    print('=' * 100)
    print(f"[{rs[0]['task']}] Q:   {rs[0]['question']}")
    print(f"        REF: {rs[0]['answer']}")
    for r in rs:
        kept = 'all 33' if r['cond'] == 'full' else r['kept']
        print(f"  {r['cond']:<12} kept={str(kept):<16} {r['gen_tokens']:>3} tok  {r['llm_latency_s']:.2f}s | {r['pred']}")

[instant] Q:   Does the file path update immediately after dropping the icon?
        REF: Yes, the location field in the save dialog instantly changes to "Secret Project," which matches the name of the folder that was dragged in.
  full         kept=all 33            15 tok  3.53s | No, the file path does not update immediately after dropping the icon.
  gate@0.9     kept=[0, 24, 28, 32]   15 tok  1.09s | No, the file path does not update immediately after dropping the icon.
  uniform@0.9  kept=[0, 11, 21, 32]   15 tok  1.05s | No, the file path does not update immediately after dropping the icon.
  pixel@0.9    kept=[0, 25, 28, 32]   15 tok  1.06s | No, the file path does not update immediately after dropping the icon.
[instant] Q:   What exercise is the person doing at the gym?
        REF: After her cardio warm-up, the video shows the person at the gym using a leg press machine for strength training.
  full         kept=all 33            15 tok  3.13s | The person is not doing an e

In [ ]:
!python src/run.py --ann PhoStream/dataset/phostream_annotations.json --video_root videos \
  --n 150 --k 3 --taus 0.88 0.90 0.93 0.95 --tasks instant backward \
  --keep_current --min_keep 4 \
  --out /content/drive/MyDrive/OmniEyes/iteration3/results_v3_k3.jsonl

2770 eligible QAs; running 150
Loading weights: 100% 1442/1442 [00:38<00:00, 37.07it/s]
[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: aud

In [ ]:
!python src/run.py --ann PhoStream/dataset/phostream_annotations.json --video_root videos \
  --n 150 --k 3 --taus 0.88 0.90 0.93 0.95 --tasks instant backward --no_audio \
  --keep_current --min_keep 4 \
  --out /content/drive/MyDrive/OmniEyes/iteration3/results_v3_k3_noaudio.jsonl

2770 eligible QAs; running 150
Loading weights: 100% 1442/1442 [00:39<00:00, 36.46it/s]
[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: aud

In [6]:
!python src/semantic_judge.py /content/drive/MyDrive/OmniEyes/iteration3/results_v3_k3.jsonl /content/drive/MyDrive/OmniEyes/iteration3/judged_v3_k3.jsonl --nli_model MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli

modules.json: 100% 349/349 [00:00<00:00, 1.43MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 760kB/s]
README.md: 100% 11.6k/11.6k [00:00<00:00, 37.9MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 183kB/s]
config.json: 100% 571/571 [00:00<00:00, 3.83MB/s]

model.safetensors: downloading bytes:  27% 117M/438M [00:01<00:01, 174MB/s, 8.68MB/s  ] 
model.safetensors: downloading bytes:  63% 276M/438M [00:01<00:00, 265MB/s, 22.8MB/s  ]
model.safetensors: downloading bytes:  95% 414M/438M [00:02<00:00, 256MB/s, 36.3MB/s  ]
model.safetensors: downloading bytes: 100% 414M/414M [00:04<00:00, 84.1MB/s, 37.2MB/s  ]
model.safetensors: reconstructing file: 100% 438M/438M [00:04<00:00, 89.0MB/s, 36.4MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 14372.71it/s]
tokenizer_config.json: 100% 363/363 [00:00<00:00, 1.84MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 28.6MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 60.7MB/s]
special_tokens_map.json: 100% 239/239 [00:

In [7]:
!python src/semantic_judge.py /content/drive/MyDrive/OmniEyes/iteration3/results_v3_k3_noaudio.jsonl /content/drive/MyDrive/OmniEyes/iteration3/judged_v3_k3_noaudio.jsonl --nli_model MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli

Loading weights: 100% 199/199 [00:00<00:00, 4825.86it/s]
Batches: 100% 8/8 [00:01<00:00,  5.71it/s]
Loading weights: 100% 394/394 [00:00<00:00, 17752.81it/s]
wrote 1950 rows -> /content/drive/MyDrive/OmniEyes/iteration3/judged_v3_k3_noaudio.jsonl

mean per condition:
cond               score       sim     p_ent     p_con     p_neu      echo   ent_con p_ent_rev
full              29.179     0.542     0.199     0.500     0.301     0.141    -0.301     0.013
gate@0.88         23.756     0.550     0.135     0.585     0.280     0.120    -0.450     0.004
gate@0.9          23.808     0.552     0.135     0.586     0.278     0.120    -0.451     0.004
gate@0.93         25.367     0.544     0.154     0.567     0.280     0.118    -0.413     0.004
gate@0.95         25.197     0.543     0.149     0.569     0.281     0.120    -0.420     0.004
pixel@0.88        23.266     0.545     0.132     0.590     0.278     0.119    -0.457     0.004
pixel@0.9         23.232     0.548     0.132     0.590     0.278   